# Table detection: how much training data do we need?

We train the same small model (YOLO11n) on **3,000, 10,000 and 18,000** pages and
score each one on the same 500 validation pages. Everything else stays fixed, so
any difference comes from the amount of data.

| Held fixed | Value |
|---|---|
| Model | `yolo11n.pt` (pretrained on COCO) |
| Epochs / early-stopping patience | 20 / 5 |
| Image size, batch, seed | 640 px, 8, 42 |
| Augmentation | no mosaic, no flips |
| Validation / test pages | the same 500 / 500 for every run |

The training sets are **nested**: the 3,000 pages are the first 3,000 of the
10,000, which are the first 10,000 of the 18,000. So a bigger set only *adds*
pages and the comparison is fair.

**How to run:** Kaggle notebook with **GPU** and **Internet** on, then
*Save Version → Save & Run All*. Everything in `/kaggle/working` is saved with
the version.

## 1. Install

In [ ]:
%pip install -q "ultralytics==8.3.221" "pyarrow==21.0.0" "huggingface_hub==0.36.0"

## 2. Imports

In [ ]:
import json
import shutil
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq
import torch
import yaml
from huggingface_hub import HfFileSystem
from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO, settings

## 3. Settings

In [ ]:
# --- What we vary ------------------------------------------------------------
TRAIN_SIZES = [3000, 10000, 18000]

# --- Held fixed for every run ------------------------------------------------
PRETRAINED = "yolo11n.pt"
EPOCHS = 20
PATIENCE = 5
IMGSZ = 640
BATCH = 8
SEED = 42
PAGES_TRAIN = max(TRAIN_SIZES)  # Select once; smaller sets are the first N pages.
PAGES_VALIDATION = 500
PAGES_TEST = 500
MAX_PAGES_PER_DOCUMENT = 10

# --- Data source -------------------------------------------------------------
REPO = "docling-project/DocLayNet-v1.2"
REVISION = "85cadd47c3ac59e1a2c3677ec72a02b143d96430"  # Pinned so the data never changes.
TABLE_CATEGORY_ID = 9  # "Table" in DocLayNet.
TABLE_CLASS_ID = 0     # "table" in our one-class dataset.

# --- Folders -----------------------------------------------------------------
DATA = Path("/kaggle/temp/table_data")   # Downloaded pages. Not saved.
OUTPUT = Path("/kaggle/working/scaling") # Models and results. Saved with the version.
WORKERS = 2                              # Kaggle machines have few CPU cores.

DATA.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
fs = HfFileSystem()
ROOT = f"datasets/{REPO}@{REVISION}"

settings.update({"wandb": False, "mlflow": False, "comet": False, "clearml": False})
assert torch.cuda.is_available(), "Turn on a GPU in the notebook settings."
print("GPU:", torch.cuda.get_device_name(0))

## 4. List every page in the dataset

Reads only the page metadata and annotation classes, not the images. For every
page we learn: which file it is in, which document it came from, its document
type, and how many tables it has.

In [ ]:
def build_index(split):
    pages = []
    files = sorted(fs.glob(f"{ROOT}/data/{split}-*.parquet"))

    for file in tqdm(files, desc=f"Index {split}"):
        with fs.open(file, "rb", block_size=64 * 1024) as stream:  # Small reads: two thin columns.
            parquet = pq.ParquetFile(stream, pre_buffer=False)

            for group in range(parquet.num_row_groups):
                rows = parquet.read_row_group(
                    group, columns=["metadata", "category_id"], use_threads=False
                ).to_pylist()

                for row_number, row in enumerate(rows):
                    meta = row["metadata"]
                    pages.append({
                        "file": file,
                        "group": group,
                        "row": row_number,
                        "page_hash": meta["page_hash"],
                        "domain": meta["doc_category"],
                        "document": f"{meta['collection']}/{meta['original_filename']}",
                        "tables": row["category_id"].count(TABLE_CATEGORY_ID),
                    })
    return pd.DataFrame(pages)

## 5. Choose the pages

We pick the pages in two steps:

1. **Shuffle, then keep at most 10 pages per document**, so one long report
   cannot dominate.
2. **Stratify** by document type and "has a table / has no table", so every kind
   of page is represented in proportion to its size.

In [ ]:
def allocate_quotas(group_sizes, total):
    # One place per group, the rest shared in proportion to group size.
    quotas = pd.Series(1, index=group_sizes.index)
    spare = group_sizes - 1
    exact_share = (total - len(group_sizes)) * spare / max(spare.sum(), 1)
    quotas = quotas + exact_share.astype(int)

    # Rounding down leaves a few places: give them to the largest remainders.
    leftover = total - quotas.sum()
    if leftover > 0:
        remainders = (exact_share - exact_share.astype(int))[quotas < group_sizes]
        quotas[remainders.nlargest(leftover).index] += 1
    return quotas


def select_pages(index, count):
    pool = index.sample(frac=1, random_state=SEED)
    pool = pool.groupby("document", sort=False).head(MAX_PAGES_PER_DOCUMENT)
    pool = pool.assign(has_table=pool["tables"] > 0)

    groups = pool.groupby(["domain", "has_table"], sort=True)
    quotas = allocate_quotas(groups.size(), count)
    selected = pd.concat([pages.head(quotas[name]) for name, pages in groups])

    assert len(selected) == count
    return selected

In [ ]:
pages = {
    "train": select_pages(build_index("train"), PAGES_TRAIN).sample(frac=1, random_state=SEED),
    "validation": select_pages(build_index("validation"), PAGES_VALIDATION),
    "test": select_pages(build_index("test"), PAGES_TEST),
}

for split, selected in pages.items():
    print(f"{split:<11} {len(selected):>6,} pages, {(selected['tables'] > 0).mean():.0%} with a table")

### Checks before downloading

1. **No document appears in two splits.** Otherwise the model could score well
   by memorising a document's layout.
2. **Same pages as the earlier run** (optional). Add `selected_pages.json` from
   an earlier run as a Kaggle dataset to run this check.

In [ ]:
documents = {split: set(selected["document"]) for split, selected in pages.items()}
assert documents["train"].isdisjoint(documents["validation"])
assert documents["train"].isdisjoint(documents["test"])
assert documents["validation"].isdisjoint(documents["test"])
print("No document is shared between splits.")

previous = list(Path("/kaggle/input").rglob("selected_pages.json"))
if previous:
    old = json.loads(previous[0].read_text())
    for split, selected in pages.items():
        assert {r["page_hash"] for r in old[split]} == set(selected["page_hash"]), split
    print("Same pages as the earlier run.")
else:
    print("selected_pages.json not attached, skipping that check.")

## 6. Download the pages and write YOLO labels

DocLayNet boxes are pixels `[left, top, width, height]`. YOLO wants
`class center_x center_y width height`, each divided by the page size. Pages
without a table get an empty label file, so the model also learns what a
table-free page looks like.

In [ ]:
def to_yolo_labels(row, width, height):
    lines = []
    for category, (left, top, box_width, box_height) in zip(row["category_id"], row["bboxes"]):
        if category != TABLE_CATEGORY_ID:
            continue
        # Clip to the page in case an annotation runs over the edge.
        right = min(left + box_width, width)
        bottom = min(top + box_height, height)
        left, top = max(left, 0), max(top, 0)
        lines.append(f"{TABLE_CLASS_ID} "
                     f"{(left + right) / 2 / width:.6f} {(top + bottom) / 2 / height:.6f} "
                     f"{(right - left) / width:.6f} {(bottom - top) / height:.6f}")
    return lines


def download_split(selected, split):
    images_dir = DATA / "images" / split
    labels_dir = DATA / "labels" / split
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)

    image_paths = {}
    progress = tqdm(total=len(selected), desc=f"Download {split}")

    for file, file_pages in selected.groupby("file"):
        with fs.open(file, "rb", block_size=8 * 1024 * 1024) as stream:  # Big reads: whole images.
            parquet = pq.ParquetFile(stream, pre_buffer=False)

            for group, group_pages in file_pages.groupby("group"):
                block = parquet.read_row_group(
                    int(group), columns=["image", "bboxes", "category_id", "metadata"],
                    use_threads=False,
                )
                for row in block.take(group_pages["row"].tolist()).to_pylist():
                    raw = row["image"]["bytes"]
                    with Image.open(BytesIO(raw)) as image:
                        width, height = image.size
                        suffix = image.format.lower()

                    page_hash = row["metadata"]["page_hash"]
                    image_path = images_dir / f"{page_hash}.{suffix}"
                    image_path.write_bytes(raw)  # Saved as-is: no re-encoding.
                    labels = to_yolo_labels(row, width, height)
                    (labels_dir / f"{page_hash}.txt").write_text("\n".join(labels))

                    image_paths[page_hash] = str(image_path)
                    progress.update()
    progress.close()

    # Keep the order of `selected`, not the order the files were read in.
    return [image_paths[page_hash] for page_hash in selected["page_hash"]]

In [ ]:
image_lists = {split: download_split(selected, split) for split, selected in pages.items()}

Save the chosen pages, so later notebooks can check they use exactly the same ones.

In [ ]:
columns = ["page_hash", "domain", "document", "tables"]
saved = {split: selected[columns].to_dict("records") for split, selected in pages.items()}
(OUTPUT / "selected_pages.json").write_text(json.dumps(saved))

## 7. Write one dataset config per training size

Every config points at the same validation and test pages. Only the training
list changes: the first 3,000, 10,000 or 18,000 pages.

In [ ]:
(DATA / "validation.txt").write_text("\n".join(image_lists["validation"]))
(DATA / "test.txt").write_text("\n".join(image_lists["test"]))

data_configs = {}
for size in TRAIN_SIZES:
    (DATA / f"train_{size}.txt").write_text("\n".join(image_lists["train"][:size]))

    data_configs[size] = DATA / f"data_{size}.yaml"
    data_configs[size].write_text(yaml.safe_dump({
        "path": str(DATA),
        "train": f"train_{size}.txt",
        "val": "validation.txt",
        "test": "test.txt",
        "names": {TABLE_CLASS_ID: "table"},
    }))

print(data_configs[TRAIN_SIZES[0]].read_text())

## 8. Train and score one model per training size

- `epochs=20, patience=5`: up to 20 epochs; stop early if validation mAP has
  not improved for 5 epochs in a row.
- `mosaic=0, fliplr=0, flipud=0`: these augmentations stitch pages together or
  mirror them. Real documents never look like that.

After training, each model is scored on the 500 validation pages with the same
settings, so the numbers can be compared directly.

In [ ]:
def evaluate(weights, data_config, split, name):
    metrics = YOLO(str(weights)).val(
        data=str(data_config),
        split=split,
        imgsz=IMGSZ,
        batch=4,
        device=0,
        half=False,
        rect=False,
        conf=0.001,  # Keep low-confidence boxes so the PR curve is complete.
        project=str(OUTPUT / "runs"),
        name=name,
        exist_ok=True,
    )
    return {
        "mAP50": round(float(metrics.box.map50), 4),
        "mAP50_95": round(float(metrics.box.map), 4),
        "precision": round(float(metrics.box.mp), 4),
        "recall": round(float(metrics.box.mr), 4),
    }

In [ ]:
results = {}

for size in TRAIN_SIZES:
    print(f"\n===== Training on {size:,} pages =====")
    model = YOLO(PRETRAINED)
    model.train(
        data=str(data_configs[size]),
        epochs=EPOCHS,
        patience=PATIENCE,
        imgsz=IMGSZ,
        batch=BATCH,
        seed=SEED,
        device=0,
        workers=WORKERS,
        mosaic=0.0,
        fliplr=0.0,
        flipud=0.0,
        project=str(OUTPUT / "runs"),
        name=f"train_{size}",
        exist_ok=True,
    )

    best_weights = OUTPUT / f"best_{size}.pt"
    shutil.copy2(Path(model.trainer.save_dir) / "weights" / "best.pt", best_weights)

    results[size] = evaluate(best_weights, data_configs[size], "val", f"val_{size}")
    print(f"{size:,} pages:", results[size])

    # Saved after every run, so a crash later does not lose finished results.
    (OUTPUT / "scaling_results.json").write_text(json.dumps(results, indent=2))

## 9. Results

In [ ]:
table = pd.DataFrame(results).T
table.index.name = "train pages"
table

### Scaling curve

The x-axis is logarithmic: each step is roughly 3× more data. If the line is
still rising at 18,000 pages, more data would probably still help.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(table.index, table["mAP50"], marker="o", label="mAP50")
plt.plot(table.index, table["mAP50_95"], marker="o", label="mAP50-95")

axis = plt.gca()
axis.set_xscale("log")
axis.minorticks_off()  # Hide the 4x10^3-style log ticks; only our three sizes are labelled.
axis.set_xticks(TRAIN_SIZES)
axis.set_xticklabels([f"{size:,}" for size in TRAIN_SIZES])

plt.xlabel("training pages")
plt.ylabel("validation score")
plt.title("More training pages, better table detection")
plt.grid(alpha=0.3)
plt.legend()
plt.savefig(OUTPUT / "scaling_curve.png", dpi=300, bbox_inches="tight")
plt.show()

### Did the runs converge?

If a curve is still climbing at the last epoch, that model would have improved
with more epochs. Then the gap between sizes is partly a training-time effect,
not only a data effect.

In [ ]:
plt.figure(figsize=(7, 4))
for size in TRAIN_SIZES:
    history = pd.read_csv(OUTPUT / "runs" / f"train_{size}" / "results.csv")
    history.columns = history.columns.str.strip()
    plt.plot(history["epoch"], history["metrics/mAP50-95(B)"], label=f"{size:,} pages")

plt.xlabel("epoch")
plt.ylabel("validation mAP50-95")
plt.title("Validation mAP50-95 per epoch")
plt.grid(alpha=0.3)
plt.legend()
plt.savefig(OUTPUT / "training_curves.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Test score of the largest model (optional)

The test pages are kept aside for the final report. Turn this on only if you
want a test number for the 18,000-page model.

In [ ]:
RUN_TEST = False

if RUN_TEST:
    largest = max(TRAIN_SIZES)
    test_scores = evaluate(OUTPUT / f"best_{largest}.pt", data_configs[largest], "test", f"test_{largest}")
    (OUTPUT / "test_results.json").write_text(json.dumps({largest: test_scores}, indent=2))
    print(json.dumps(test_scores, indent=2))
else:
    print("Skipped. Set RUN_TEST = True to score on the test pages.")